# 01 · Load and prepare KBO Open Data

**Project question.** Which addresses and contact clusters in the Antwerp region show patterns associated with front or shell company activity, and how many of these survive manual verification?

This notebook does the groundwork only:

1. load one full KBO Open Data extract into DuckDB;
2. check the files against the expected schema;
3. build clean core tables (enterprises, names, main activity);
4. normalise addresses and contact details for the study area;
5. build the enterprise–address–contact link table and a first network;
6. write everything to `data/processed/` for the next notebooks.

**Data and licence.** KBO Open Data (FOD Economie) requires registration and acceptance of the licence. The raw files are *not* part of this repository. Sole proprietorships (eenmanszaken) appear under a natural person's name, so part of the data is personal data under the GDPR. With `PUBLIC_MODE = True` the notebook never displays names, enterprise numbers or street addresses, only pseudonymised keys. Strip outputs before committing anyway (`nbstripout`).

**Status of the indicators.** Nothing in this notebook says anything about criminal activity. Concentrations and shared contact details have many legitimate explanations (accountancy offices, business centres, franchises, holdings, family firms). The point of later notebooks is to separate these, not to produce a list of suspects.

## 0 · Configuration

In [ ]:
from pathlib import Path
import hashlib, zipfile, warnings

import duckdb
import pandas as pd
import networkx as nx

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Project root: works whether the notebook runs from the repo root or from notebooks/
PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT / "data" / "raw"          # put KboOpenData_XXXX_YYYY_MM_DD_Full.zip here
INTERIM = PROJECT / "data" / "interim"      # unzipped CSVs + DuckDB file
PROCESSED = PROJECT / "data" / "processed"  # parquet outputs for later notebooks
for d in (RAW_DIR, INTERIM, PROCESSED):
    d.mkdir(parents=True, exist_ok=True)

# Study area (Belgian postcodes): province of Antwerp = 2000-2999,
# plus the left-bank port area in East Flanders (Waaslandhaven): 9120 Beveren, 9130 Kallo/Doel.
STUDY_POSTCODES = [str(p) for p in range(2000, 3000)] + ["9120", "9130"]

# Addresses with more enterprises than this are treated as hubs (business centres,
# domiciliation offices) and left out of the network projection; they are analysed separately.
HUB_THRESHOLD = 50

# Never show names, enterprise numbers or street addresses in notebook output.
PUBLIC_MODE = True

LABEL_LANG = "NL"

In [ ]:
def pseudo(value, prefix=""):
    """Stable short hash for display. Not anonymisation: keys stay linkable inside the project."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    return prefix + hashlib.sha256(str(value).encode()).hexdigest()[:10]

def show(df, sensitive_cols=(), n=20):
    """Display helper that pseudonymises sensitive columns in PUBLIC_MODE."""
    out = df.head(n).copy()
    if PUBLIC_MODE:
        for c in sensitive_cols:
            if c in out.columns:
                out[c] = out[c].map(lambda v: pseudo(v, c[:3] + "_"))
    return out

def q(sql):
    return con.execute(sql).df()

## 1 · Locate and unzip the extract

In [ ]:
zips = sorted(RAW_DIR.glob("KboOpenData_*_Full.zip"))
assert zips, f"No full extract found. Download one from the KBO Open Data portal into {RAW_DIR}"
ZIP_PATH = zips[-1]   # extract numbers are zero-padded, so the last one is the newest
EXTRACT_DIR = INTERIM / ZIP_PATH.stem
if not EXTRACT_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)
print("Using", ZIP_PATH.name)
sorted(p.name for p in EXTRACT_DIR.glob("*.csv"))

## 2 · Load into DuckDB

Every file is loaded as text first (`all_varchar`), so nothing is silently coerced. Types are set explicitly in step 4. The DuckDB file in `data/interim/` means later notebooks can reuse the tables without reloading.

In [ ]:
con = duckdb.connect(str(INTERIM / "kbo.duckdb"))

FILES = ["meta", "code", "enterprise", "establishment", "denomination",
         "address", "contact", "activity", "branch"]

for name in FILES:
    path = EXTRACT_DIR / f"{name}.csv"
    if not path.exists():
        warnings.warn(f"{name}.csv missing from extract")
        continue
    con.execute(f"""
        CREATE OR REPLACE TABLE raw_{name} AS
        SELECT * FROM read_csv('{path.as_posix()}', header=true, all_varchar=true,
                               delim=',', quote='"')
    """)

pd.DataFrame(
    [(n, con.execute(f"SELECT count(*) FROM raw_{n}").fetchone()[0]) for n in FILES
     if (EXTRACT_DIR / f"{n}.csv").exists()],
    columns=["table", "rows"],
)

In [ ]:
q("SELECT * FROM raw_meta")

## 3 · Schema check

The expected columns follow the KBO Open Data cookbook. If FOD Economie changes the format, this cell tells you where before anything downstream breaks.

In [ ]:
EXPECTED = {
    "enterprise":    ["EnterpriseNumber", "Status", "JuridicalSituation", "TypeOfEnterprise",
                      "JuridicalForm", "JuridicalFormCAC", "StartDate"],
    "establishment": ["EstablishmentNumber", "StartDate", "EnterpriseNumber"],
    "denomination":  ["EntityNumber", "Language", "TypeOfDenomination", "Denomination"],
    "address":       ["EntityNumber", "TypeOfAddress", "CountryNL", "CountryFR", "Zipcode",
                      "MunicipalityNL", "MunicipalityFR", "StreetNL", "StreetFR",
                      "HouseNumber", "Box", "ExtraAddressInfo", "DateStrikingOff"],
    "contact":       ["EntityNumber", "EntityContact", "ContactType", "Value"],
    "activity":      ["EntityNumber", "ActivityGroup", "NaceVersion", "NaceCode", "Classification"],
    "branch":        ["Id", "StartDate", "EnterpriseNumber"],
    "code":          ["Category", "Code", "Language", "Description"],
}
problems = []
for t, cols in EXPECTED.items():
    actual = [r[0] for r in con.execute(f"DESCRIBE raw_{t}").fetchall()]
    missing, extra = set(cols) - set(actual), set(actual) - set(cols)
    if missing or extra:
        problems.append((t, sorted(missing), sorted(extra)))
pd.DataFrame(problems, columns=["table", "missing", "unexpected"]) if problems else "Schema matches."

A few distributions worth knowing before building anything:

In [ ]:
display(q("SELECT Status, count(*) n FROM raw_enterprise GROUP BY 1 ORDER BY n DESC"))
display(q("SELECT TypeOfEnterprise, count(*) n FROM raw_enterprise GROUP BY 1 ORDER BY n DESC"))
display(q("SELECT TypeOfAddress, count(*) n FROM raw_address GROUP BY 1 ORDER BY n DESC"))
display(q("SELECT ContactType, EntityContact, count(*) n FROM raw_contact GROUP BY 1,2 ORDER BY n DESC"))
display(q("SELECT NaceVersion, Classification, count(*) n FROM raw_activity GROUP BY 1,2 ORDER BY 1,2"))

**Source-critical note.** The extract only carries the *current* state of active entities: no history, no stopped companies. Churn (companies appearing and disappearing at one address) therefore cannot be measured from one extract. It needs either the monthly update files collected over time, or start-date cohorts as a proxy, which is what this notebook uses.

## 4 · Core tables

In [ ]:
# Reference date for "recent" start dates: the snapshot date in meta.csv, falling back to today.
snap = con.execute("""
    SELECT try_strptime(Value, '%d-%m-%Y')::DATE FROM raw_meta WHERE Variable = 'SnapshotDate'
""").fetchone()
SNAPSHOT = pd.Timestamp(snap[0]) if snap and snap[0] else pd.Timestamp.today().normalize()
print("Snapshot date:", SNAPSHOT.date())

con.execute(f"""
CREATE OR REPLACE TABLE code_lbl AS
SELECT Category, Code, Description FROM raw_code WHERE Language = '{LABEL_LANG}';

CREATE OR REPLACE TABLE enterprise AS
SELECT e.EnterpriseNumber                       AS enterprise_id,
       e.Status                                 AS status,
       e.JuridicalSituation                     AS juridical_situation,
       e.TypeOfEnterprise = '1'                 AS is_natural_person,
       e.JuridicalForm                          AS juridical_form,
       jf.Description                           AS juridical_form_label,
       try_strptime(e.StartDate, '%d-%m-%Y')::DATE AS start_date
FROM raw_enterprise e
LEFT JOIN code_lbl jf ON jf.Category = 'JuridicalForm' AND jf.Code = e.JuridicalForm;

-- Every entity that can carry an address or contact, mapped back to its enterprise
CREATE OR REPLACE TABLE entity_map AS
SELECT EnterpriseNumber AS entity_id, EnterpriseNumber AS enterprise_id, 'enterprise' AS entity_kind FROM raw_enterprise
UNION ALL
SELECT EstablishmentNumber, EnterpriseNumber, 'establishment' FROM raw_establishment
UNION ALL
SELECT Id, EnterpriseNumber, 'branch' FROM raw_branch;

-- One display name per enterprise: official name (001) first, Dutch preferred
CREATE OR REPLACE TABLE enterprise_name AS
SELECT enterprise_id, name FROM (
  SELECT m.enterprise_id, d.Denomination AS name,
         row_number() OVER (PARTITION BY m.enterprise_id
                            ORDER BY d.TypeOfDenomination = '001' DESC,
                                     d.Language = '2' DESC,
                                     d.TypeOfDenomination) AS rk
  FROM raw_denomination d JOIN entity_map m ON m.entity_id = d.EntityNumber
) WHERE rk = 1;

-- Main activity per enterprise: MAIN classification, newest NACE version, enterprise level first
CREATE OR REPLACE TABLE enterprise_main_nace AS
SELECT enterprise_id, nace_version, nace_code, left(nace_code, 2) AS nace_division FROM (
  SELECT m.enterprise_id, a.NaceVersion AS nace_version, a.NaceCode AS nace_code,
         row_number() OVER (PARTITION BY m.enterprise_id
                            ORDER BY a.NaceVersion DESC,
                                     m.entity_kind = 'enterprise' DESC,
                                     a.ActivityGroup) AS rk
  FROM raw_activity a JOIN entity_map m ON m.entity_id = a.EntityNumber
  WHERE a.Classification = 'MAIN'
) WHERE rk = 1;
""")

q("""
SELECT count(*) AS enterprises,
       avg(is_natural_person::INT) AS share_natural_person,
       avg((juridical_form_label IS NULL AND juridical_form IS NOT NULL)::INT) AS unlabelled_forms,
       min(start_date) AS earliest_start, max(start_date) AS latest_start
FROM enterprise
""")

In [ ]:
# Check the NACE code labels join (category names in code.csv are e.g. 'Nace2008', 'Nace2025').
q("""
SELECT n.nace_version, count(*) AS enterprises,
       avg((c.Description IS NOT NULL)::INT) AS share_with_label
FROM enterprise_main_nace n
LEFT JOIN code_lbl c ON c.Category = 'Nace' || n.nace_version AND c.Code = n.nace_code
GROUP BY 1 ORDER BY 1
""")

## 5 · Addresses in the study area

Normalisation is deliberately conservative and transparent: upper case, accents stripped, punctuation removed, a few abbreviations expanded, leading zeros removed from house numbers. Two keys are built:

- `building_key` = postcode + street + house number. This is the unit for concentration analysis (a letterbox address is usually a building, not a box).
- `unit_key` = building key + box.

Fuzzy matching and linking to the national address register (BeST) are left for a later notebook. The match rate here is the baseline to beat.

In [ ]:
STUDY_LIST = ",".join(f"'{p}'" for p in STUDY_POSTCODES)

con.execute(f"""
CREATE OR REPLACE MACRO norm_text(s) AS
  trim(regexp_replace(regexp_replace(upper(strip_accents(coalesce(s, ''))),
       '[^A-Z0-9 ]', ' ', 'g'), '\\s+', ' ', 'g'));

CREATE OR REPLACE MACRO norm_street(s) AS
  trim(regexp_replace(regexp_replace(regexp_replace(regexp_replace(norm_text(s),
       '(STWG|STWEG)\\b', 'STEENWEG', 'g'),
       '\\bLN\\b', 'LAAN', 'g'),
       '\\bPL\\b', 'PLEIN', 'g'), 'STR\\b', 'STRAAT', 'g'));

CREATE OR REPLACE MACRO norm_number(s) AS
  regexp_replace(replace(upper(coalesce(s, '')), ' ', ''), '^0+', '');

CREATE OR REPLACE MACRO norm_box(s) AS
  regexp_replace(regexp_replace(upper(replace(coalesce(s, ''), ' ', '')),
       '^(BUS|BTE|BOITE|BOX|B)', ''), '^0+', '');

CREATE OR REPLACE TABLE address_study AS
SELECT m.enterprise_id,
       a.EntityNumber                         AS entity_id,
       m.entity_kind,
       a.TypeOfAddress                        AS address_type,
       a.Zipcode                              AS postcode,
       coalesce(nullif(a.MunicipalityNL, ''), a.MunicipalityFR) AS municipality,
       norm_street(coalesce(nullif(a.StreetNL, ''), a.StreetFR)) AS street_norm,
       norm_number(a.HouseNumber)             AS house_norm,
       norm_box(a.Box)                        AS box_norm
FROM raw_address a
JOIN entity_map m ON m.entity_id = a.EntityNumber
WHERE a.Zipcode IN ({STUDY_LIST})
  AND coalesce(a.CountryNL, '') IN ('', 'België')
  AND coalesce(a.DateStrikingOff, '') = '';

ALTER TABLE address_study ADD COLUMN building_key VARCHAR;
ALTER TABLE address_study ADD COLUMN unit_key VARCHAR;
UPDATE address_study SET
  building_key = postcode || '|' || street_norm || '|' || house_norm,
  unit_key     = postcode || '|' || street_norm || '|' || house_norm || '|' || box_norm;
""")

q("""
SELECT count(*) AS address_rows,
       count(DISTINCT enterprise_id) AS enterprises,
       count(DISTINCT building_key) AS buildings,
       avg((street_norm = '' OR house_norm = '')::INT) AS share_incomplete
FROM address_study
""")

### Address-level summary

Per building: how many enterprises, how many started recently, how many are sole proprietorships, and how many different NACE divisions are present. Diverse, recent, legal-person-heavy concentrations at an ordinary address are what later scoring will look at.

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE address_summary AS
SELECT s.building_key,
       any_value(s.postcode)                                   AS postcode,
       any_value(s.municipality)                               AS municipality,
       count(DISTINCT s.enterprise_id)                         AS n_enterprises,
       count(DISTINCT s.enterprise_id) FILTER (WHERE e.is_natural_person)   AS n_natural_person,
       count(DISTINCT s.enterprise_id) FILTER (WHERE e.start_date >= DATE '{SNAPSHOT.date()}' - INTERVAL 3 YEAR)
                                                               AS n_started_last_3y,
       count(DISTINCT n.nace_division)                         AS n_nace_divisions,
       count(DISTINCT s.unit_key)                              AS n_boxes
FROM address_study s
JOIN enterprise e USING (enterprise_id)
LEFT JOIN enterprise_main_nace n USING (enterprise_id)
WHERE s.street_norm <> '' AND s.house_norm <> ''
GROUP BY s.building_key;
""")

display(q("""
SELECT n_bucket, count(*) AS buildings FROM (
  SELECT CASE WHEN n_enterprises = 1 THEN '1'
              WHEN n_enterprises <= 5 THEN '2-5'
              WHEN n_enterprises <= 20 THEN '6-20'
              WHEN n_enterprises <= {h} THEN '21-{h}'
              ELSE '>{h} (hub)' END AS n_bucket, n_enterprises
  FROM address_summary)
GROUP BY 1 ORDER BY min(n_enterprises)
""".format(h=HUB_THRESHOLD)))

show(q("""
SELECT building_key, postcode, municipality, n_enterprises, n_started_last_3y,
       n_natural_person, n_nace_divisions, n_boxes
FROM address_summary ORDER BY n_enterprises DESC LIMIT 25
"""), sensitive_cols=["building_key"], n=25)

## 6 · Contact details

Phone numbers are reduced to digits in international form (`32…`), e-mail addresses lower-cased, websites reduced to the domain. A contact detail shared by several *unrelated* enterprises is a classic linking signal, but so is an accountant's office number. That distinction is for the next notebook.

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE contact_study AS
WITH c AS (
  SELECT m.enterprise_id, upper(c.ContactType) AS contact_type, trim(c.Value) AS raw_value,
         regexp_replace(c.Value, '[^0-9]', '', 'g') AS digits
  FROM raw_contact c JOIN entity_map m ON m.entity_id = c.EntityNumber
  WHERE m.enterprise_id IN (SELECT DISTINCT enterprise_id FROM address_study)
),
n AS (
  SELECT enterprise_id, contact_type,
    CASE
      WHEN contact_type IN ('TEL', 'FAX', 'GSM') THEN
        CASE WHEN digits LIKE '00%' THEN substr(digits, 3)
             WHEN digits LIKE '0%'  THEN '32' || substr(digits, 2)
             ELSE digits END
      WHEN contact_type = 'EMAIL' THEN lower(raw_value)
      WHEN contact_type = 'WEB' THEN
        split_part(regexp_replace(lower(raw_value), '^(https?://)?(www\\.)?', ''), '/', 1)
      ELSE lower(raw_value)
    END AS value_norm
  FROM c
)
SELECT DISTINCT enterprise_id, contact_type, value_norm,
       contact_type || '|' || value_norm AS contact_key
FROM n WHERE coalesce(value_norm, '') <> '';

CREATE OR REPLACE TABLE shared_contacts AS
SELECT contact_key, contact_type, count(DISTINCT enterprise_id) AS n_enterprises
FROM contact_study GROUP BY 1, 2 HAVING count(DISTINCT enterprise_id) > 1;
""")

display(q("""
SELECT contact_type, count(*) AS rows, count(DISTINCT enterprise_id) AS enterprises
FROM contact_study GROUP BY 1 ORDER BY 2 DESC
"""))
display(q("""
SELECT contact_type, count(*) AS shared_values, max(n_enterprises) AS max_enterprises_per_value,
       quantile_cont(n_enterprises, 0.5) AS median
FROM shared_contacts GROUP BY 1
"""))
show(q("SELECT * FROM shared_contacts ORDER BY n_enterprises DESC LIMIT 15"),
     sensitive_cols=["contact_key"])

## 7 · Link table and first network

Bipartite links: enterprise → building and enterprise → shared contact detail. Hub addresses (more than `HUB_THRESHOLD` enterprises) are excluded from the projection, otherwise every business centre collapses hundreds of unrelated firms into one component. The connected components that remain are the candidate clusters for the next notebook.

In [ ]:
links = q(f"""
SELECT DISTINCT s.enterprise_id, 'A:' || s.building_key AS node, 'address' AS link_type
FROM address_study s
JOIN address_summary a USING (building_key)
WHERE a.n_enterprises BETWEEN 2 AND {HUB_THRESHOLD}
UNION ALL
SELECT DISTINCT c.enterprise_id, 'C:' || c.contact_key, 'contact'
FROM contact_study c JOIN shared_contacts s USING (contact_key)
WHERE s.n_enterprises <= {HUB_THRESHOLD}
""")

G = nx.Graph()
G.add_nodes_from(("E:" + e for e in links.enterprise_id.unique()), kind="enterprise")
G.add_nodes_from(links.node.unique(), kind="link")
G.add_edges_from(zip("E:" + links.enterprise_id, links.node))

comps = list(nx.connected_components(G))
components = []
for i, comp in enumerate(comps):
    ents = [n[2:] for n in comp if n.startswith("E:")]
    if len(ents) < 2:
        continue
    components.append({
        "component_id": i,
        "n_enterprises": len(ents),
        "n_addresses": sum(n.startswith("A:") for n in comp),
        "n_shared_contacts": sum(n.startswith("C:") for n in comp),
    })
components = pd.DataFrame(components).sort_values("n_enterprises", ascending=False)

print(f"{len(links):,} links, {G.number_of_nodes():,} nodes, {len(components):,} components with 2+ enterprises")
display(components.n_enterprises.describe())
# Most interesting at this stage: components that are tied together by BOTH addresses and contacts
components.query("n_addresses >= 2 and n_shared_contacts >= 1").head(20)

In [ ]:
membership = pd.DataFrame(
    [(n[2:], cid) for cid, comp in enumerate(comps)
     for n in comp if n.startswith("E:")],
    columns=["enterprise_id", "component_id"],
)

## 8 · Write outputs

In [ ]:
outputs = {
    "enterprise": "SELECT e.*, n.nace_code, n.nace_division, n.nace_version FROM enterprise e "
                  "LEFT JOIN enterprise_main_nace n USING (enterprise_id) "
                  "WHERE enterprise_id IN (SELECT DISTINCT enterprise_id FROM address_study)",
    "enterprise_name": "SELECT * FROM enterprise_name "
                       "WHERE enterprise_id IN (SELECT DISTINCT enterprise_id FROM address_study)",
    "address_study": "SELECT * FROM address_study",
    "address_summary": "SELECT * FROM address_summary",
    "contact_study": "SELECT * FROM contact_study",
    "shared_contacts": "SELECT * FROM shared_contacts",
}
for name, sql in outputs.items():
    con.execute(f"COPY ({sql}) TO '{(PROCESSED / (name + '.parquet')).as_posix()}' (FORMAT parquet)")
links.to_parquet(PROCESSED / "links.parquet", index=False)
components.to_parquet(PROCESSED / "components.parquet", index=False)
membership.to_parquet(PROCESSED / "component_membership.parquet", index=False)

sorted(p.name for p in PROCESSED.glob("*.parquet"))

## Next steps

- **02 · Address quality**: link `building_key` to BeST (national address register) for coordinates and to catch spelling variants; report the match rate.
- **03 · Scoring**: transparent rule-based indicators per address and per component (concentration, recent starts, sector mix, risk sectors, residential vs. office location). Every weight explained.
- **04 · False positives**: manual classification of the top 30 hits (accountancy, business centre, franchise, holding, family firm, unexplained).
- **05 · Case studies**: two or three clusters enriched by hand from the Belgisch Staatsblad, NBB annual accounts and Street View, with a source log per step. Published pseudonymised.